# PySpark 1: foundations

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-01-foundations.ipynb)

You have a Python job that produces daily revenue on your laptop. Tomorrow the input will be split across
hundreds of files. What changes when you run the same calculation in Spark?

This module builds a DataFrame and traces it from a Python expression to a physical plan and then to tasks.
By the end you can say what runs on the driver and what runs on executors, point to a scan and an exchange in
a plan, and explain why nothing runs until an action asks for a result.

Each section explains how the feature works, where it matters and what to watch out for, then ends with a runnable example. Run it, read the output, and compare it with the notes on what to notice. Run the cells in order.

**Prerequisites:** basic [Python](../python/learning/python-01-values-and-types.ipynb) and [SQL](../sql/sql-01-basics.ipynb). A local run needs Java 17 or 21; Colab already has Java.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 pyarrow

## Decide where Spark fits

PySpark is the Python API for Apache Spark. You write DataFrame code in Python; Spark plans it and runs it in
parallel across a cluster of JVM processes. In a lakehouse it reads raw files from object storage, Kafka or
databases, cleans and joins them, and writes Delta, Iceberg or Parquet tables for analytics and machine
learning.

Spark is a heavy default, not a habit. It pays for itself once the data no longer fits one machine, once
streaming appears, or once a lot of custom Python logic has to run in parallel. Before that point a
single-node engine is faster to write, run and debug.

**Where it matters**
- **Batch ELT:** a nightly job reads a day of JSON from S3, enforces a schema, deduplicates by `order_id` and writes `main.gold.fact_order`.
- **Streaming:** the same transformations run every minute over new Kafka events.
- **Feature engineering:** one feature definition is applied to years of history for training and to today's data for scoring.

**Watch out for**
- Data that fits in memory (a few GB): pandas, Polars or DuckDB will be simpler and faster.
- Pure SQL over warehouse tables: the warehouse engine or dbt avoids shipping and operating Python.
- Point lookups and low-latency serving: Spark is a batch engine; those belong in a database.

### Example

A typical medallion flow. Each arrow is a Spark job that reads one layer and writes the next.

```text
raw files / Kafka -> bronze (typed, raw) -> silver (clean, one row per key) -> gold (aggregates) -> BI / ML
```

A good two-line interview answer: "PySpark earns its cost once data volume, streaming, or heavy custom
Python logic appears; before that, simpler engines win."

## Start a Spark session

`SparkSession` is the entry point for DataFrames, SQL, the catalog and streaming. `SparkSession.builder`
collects configuration and `getOrCreate()` returns the session already running in this process, or starts
one. On Databricks and most managed notebooks `spark` already exists; in jobs, tests and local development
you create it yourself.

Builder settings are fixed for the life of the session. Runtime SQL settings such as
`spark.sql.shuffle.partitions` can be changed later with `spark.conf.set`.

**Where it matters**
- **Local development and tests:** `master("local[2]")` runs the driver and executor in one JVM.
- **Jobs:** `spark-submit` passes the master and resources; the code only calls `getOrCreate()`.
- **Unity Catalog:** tables have three-part names, for example `spark.table("main.silver.orders")`.

**Watch out for**
- Calling `getOrCreate()` again with different builder settings silently returns the first session. Stop it first, or use `spark.conf.set` for runtime settings.
- Defaults worth knowing: `spark.sql.shuffle.partitions` is 200, `spark.sql.autoBroadcastJoinThreshold` is 10 MB, `spark.sql.files.maxPartitionBytes` is 128 MB, and AQE is on.

### Example

Start a two-core local session in UTC. The shuffle partition count is lowered because the data is tiny.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## Sample data used in every module

Every module in this series seeds the same four-table slice of an orders platform, so you can run any
module on its own and compare results across them.

| table | grain | notable columns | quirks the lessons use |
| --- | --- | --- | --- |
| `customers` | one row per customer | `region`, `signup_date` | customer 4 has no region; 7 and 8 never ordered |
| `orders` | one row per order | `status`, `order_ts`, `amount` (decimal) | order 12 lands just after midnight UTC |
| `order_items` | one row per order line | `qty`, `unit_price` | every order's lines add up to its amount |
| `events` | one row per user event | `user_id`, `event`, `ts` | one event is an exact retry |

The schemas are DDL strings: a compact way to declare names and types. Money is `decimal`, never `double`,
so totals add up exactly.

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),  # a bulk reorder
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)),
    (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)),
    (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)),
    (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)),
    (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)),
    (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),
    (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

customers.show(20)
orders.show(20)
order_items.show(17)
events.show(20)

Two invariants the later modules rely on: every order's lines reconcile with its amount, and the retried
event collapses to one row.

In [ ]:
per_order = (
    order_items
    .groupBy("order_id")
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("lines"))
    .join(orders.select("order_id", "amount"), "order_id")
)
print("orders whose lines do not add up to the amount:", per_order.where(F.col("lines") != F.col("amount")).count())
print("events:", events.count(), "| distinct events:", events.dropDuplicates().count())

## Driver, executors, jobs, stages and tasks

A PySpark application has one **driver** process that owns the `SparkSession` and one or more **executor**
processes that hold the data. The driver turns your DataFrame calls into a plan; executors run the work.

Data is split into **partitions**. An action starts a **job**. The scheduler cuts the job into **stages**
wherever rows must move between executors (a **shuffle**), and each stage runs one **task** per partition.
Operations that work row by row inside a partition, such as `filter` and `select`, are **narrow** and stay
in the same stage. Operations that need all rows with the same key together, such as `groupBy`, joins
and `distinct`, are **wide** and add a shuffle, shown as `Exchange` in a plan.

Python only runs on the driver and inside Python UDFs. The DataFrame engine itself runs in the JVM, which is
why built-in functions are fast and Python UDFs are not.

**Where it matters**
- **Memory sizing:** the driver holds the plan and anything you `collect()`; executors hold shuffle files, cached blocks and broadcast tables.
- **Debugging:** a slow aggregation shows up in the Spark UI as one shuffle stage where a few tasks take far longer than the rest.

**Watch out for**
- `collect()` and `toPandas()` move the whole result into the driver's memory.
- One action can start several jobs, for example to build a broadcast table or when adaptive execution re-plans. Read the SQL and Stages tabs together.
- `master("local[*]")` packs driver and executor into one JVM: a development tool, not a deployment topology.

### Example

The helper below captures `explain` output so a cell can assert on the plan. The filter keeps the source's
two partitions and adds no exchange; the `groupBy` adds one.

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

In [ ]:
numbers = spark.range(8, numPartitions=2)
at_least_two = numbers.where(F.col("id") >= 2)  # narrow: same stage, same partitions
by_parity = at_least_two.groupBy((F.col("id") % 2).alias("parity")).count()  # wide: needs a shuffle

print("partitions after the filter:", at_least_two.rdd.getNumPartitions())
print("--- filter plan ---")
at_least_two.explain()
print("--- groupBy plan ---")
by_parity.explain()
by_parity.show()

**What to notice:** The filter plan is a single `Range` scan with a `Filter`: no `Exchange`, so it runs in one stage and keeps
the source's two partitions. The `groupBy` plan adds `Exchange hashpartitioning(...)` between a partial and
a final `HashAggregate`: that exchange is the shuffle, and the stage boundary. Ids 2 to 7 split into three
even and three odd values.

## Work with DataFrames and columns

A DataFrame is a distributed table with a schema. You change it by returning a new DataFrame:
`select` picks and computes columns, `withColumn` adds or replaces one, `withColumnRenamed` and `drop`
reshape it, and `where` (an alias of `filter`) keeps rows. Column expressions come from
`pyspark.sql.functions`, imported as `F`; `F.col("qty")` and the plain string `"qty"` refer to the same
column, and `F.expr("qty * 2")` accepts a SQL expression.

Results come back as `Row` objects. A `Row` can be read by name (`row["sku"]`) or by attribute
(`row.sku`).

**Where it matters**
- **Readable pipelines:** a chain of small, named column expressions is easy to review and test.
- **Schema contracts:** `printSchema()` and `dtypes` show exactly which types downstream consumers receive.

**Watch out for**
- A long loop of `withColumn` calls makes a deep plan that is slow to analyse. Build many columns in one `select`.
- `show()` truncates long strings and returns nothing. Use `collect()`, `take(n)` or `first()` when you need values in Python.
- Row order is not guaranteed. Add `orderBy` before `show`, `first` or a comparison.

### Example

Derive line amounts and a bulk flag, rename and drop columns, then read values back from `Row` objects.

In [ ]:
order_lines = (
    order_items
    .withColumn("line_amount", F.col("qty") * F.col("unit_price"))
    .withColumn("is_bulk", F.expr("qty >= 10"))
    .withColumnRenamed("qty", "units")
    .drop("unit_price")
    .select("order_id", "sku", F.col("units").cast("long").alias("units"), "line_amount", "is_bulk")
)
order_lines.printSchema()
order_lines.orderBy("order_id", "sku").show(5)

first = order_lines.orderBy("order_id", "sku").first()
print(first)
print("by name:", first["sku"], "| by attribute:", first.sku)
print("bulk orders:", [row.order_id for row in order_lines.where("is_bulk").collect()])

**What to notice:** The schema shows `units` as `bigint` after the cast and `line_amount` as a decimal, because an integer
times a decimal stays exact. `unit_price` is gone and `qty` was renamed. Only order 9, the twenty-unit
reorder, is flagged as bulk.

## Prefer DataFrames to RDDs

DataFrames are schema-aware, so the **Catalyst** optimizer can see every column expression and rewrite the
plan: push filters to the file reader, read only the columns you use, and fold constants. **Tungsten** then
compiles the operators into generated Java code over a compact binary row format.

An RDD is a lower-level collection of opaque objects. Spark cannot look inside a Python lambda, and
`df.rdd` converts every row between the JVM and Python.

**Where it matters**
- **Almost all data engineering:** filters, joins, aggregations and windows belong in the DataFrame API.
- **Rare record-level logic** that no built-in function expresses is the only case for an RDD.

**Watch out for**
- `df.rdd.map(...).toDF()` for logic that `pyspark.sql.functions` already covers. You lose pushdown and column pruning, and it can be several times slower on a wide table.
- Spark Connect clients have no RDD API at all, which is one more reason to keep pipeline logic in DataFrames.

### Example

The same idea in both APIs. The DataFrame filter is visible to the optimizer; the RDD lambda runs in Python workers.

In [ ]:
paid_orders = orders.where(F.col("status") == "paid").select("order_id", "customer_id", "amount")
paid_orders.orderBy("order_id").show()

log_lines = ["GET /orders 200", "POST /orders 201", "GET /health 200"]
line_lengths = spark.sparkContext.parallelize(log_lines).map(len)
print(line_lengths.collect())

## Transformations, actions and lazy evaluation

**Transformations** (`select`, `where`, `withColumn`, `join`, `groupBy`, `agg`, `distinct`) only add steps to a
logical plan. **Actions** (`count`, `collect`, `show`, `take`, `write`) ask for a result, and only then does
Spark optimize the whole plan, cut it into stages and run tasks. Waiting lets Spark optimize the entire
query instead of each line on its own.

Each action runs the plan again from the source unless an intermediate result is cached.

**Where it matters**
- **Cheap pipeline construction:** you can build and inspect a large plan without touching any data.
- **Optimization across steps:** two filters and a projection written on separate lines become a single scan.

**Watch out for**
- Calling `count()` or `show()` after every step to "check progress" re-runs the lineage each time. Read the plan instead, or cache a frame that several actions really reuse.
- Errors in a transformation, such as a bad cast, only appear when an action runs.

### Example

Build a daily revenue query, confirm nothing ran, then trigger it twice: once with `count` and once with a write.

In [ ]:
daily_revenue = (
    orders
    .where(F.col("status").isin("paid", "shipped"))
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)
print(type(daily_revenue).__name__, "- a plan; nothing has executed yet")

print("days with revenue:", daily_revenue.count())  # action 1
daily_revenue.write.mode("overwrite").saveAsTable("daily_revenue")  # action 2 re-runs the plan

total = spark.table("daily_revenue").agg(F.sum("revenue")).first()[0]
print("revenue written:", total)

**What to notice:** The first `print` returns instantly: `daily_revenue` is only a plan. `count()` and the write each run the
whole plan from `orders`; nothing was kept in between. Paid and shipped orders add up to 2,273.90.

## Read a plan before you run it

`explain("formatted")` prints the physical plan without reading data. Read it bottom-up: the scan and what it
reads (`ReadSchema`, `PushedFilters`, `PartitionFilters`), then filters and projections, then any `Exchange`
(shuffle) and the join or aggregate strategy. Catalyst applies rule-based rewrites first (predicate pushdown,
column pruning, constant folding), then picks physical operators from table statistics.

**Where it matters**
- **Cost control:** reading 2 of 60 columns from a 2 TB Parquet table reads a fraction of the bytes, and the plan shows whether that happened.
- **Before tuning:** the plan shows whether a join broadcasts or shuffles, and how many shuffles a query has.

**Watch out for**
- With adaptive query execution on, the printed plan says `isFinalPlan=false`; Spark can still change it at runtime.
- `explain("cost")` shows the size estimates the planner used, which explains a surprising join choice.

### Example

Write `orders` as Parquet and read it back. The plan shows the `amount` filter pushed into the Parquet
reader and `status` missing from the columns it reads.

In [ ]:
orders_path = os.path.join(WORK_DIR, "orders_parquet")
orders.write.mode("overwrite").parquet(orders_path)

result = (
    spark.read.parquet(orders_path)
    .where("order_ts >= timestamp '2026-01-10'")
    .select("order_id", "customer_id", "amount")
    .where("amount > 0")
)
plan = plan_of(result)
print(plan)

**What to notice:** In the `Scan parquet` node, `PushedFilters` lists `GreaterThan(amount,0.00)` and the timestamp filter:
the Parquet reader applies them and can skip row groups. `ReadSchema` lists only `order_id`,
`customer_id`, `amount` and `order_ts`; `status` is never read. Two filters and a projection became one scan.

## Spark 4 features worth knowing

Spark 4.0 shipped in May 2025. Three additions change how pipelines are written, not just how fast they run:

- **Spark Connect:** a thin client that sends unresolved plans to a remote Spark server over gRPC, instead
  of starting a driver JVM in your Python process. DataFrame and SQL code is unchanged; only session creation
  differs.
- **`VARIANT`:** semi-structured JSON stored in a binary form that Spark can query without a fixed schema.
- **Python data sources:** custom batch and streaming sources and sinks written in pure Python.

Spark 4 also turns ANSI SQL mode on by default (invalid casts raise errors; see
[PySpark 2](pyspark-02-reading-sql-and-cleaning.ipynb)) and adds `transformWithStateInPandas` for custom
streaming state ([PySpark 6](pyspark-06-structured-streaming.ipynb)).

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote("sc://spark-connect.acme.internal:15002").getOrCreate()
# or, with no code change: export SPARK_REMOTE="sc://spark-connect.acme.internal:15002"
orders = spark.table("main.silver.orders").where("status = 'paid'")
```

**Where it matters**
- **Spark Connect:** many notebooks, IDEs and services share one cluster; a client crash cannot take the driver down, and `pip install pyspark-client` installs the client without a JVM.
- **`VARIANT`:** land raw payloads in bronze without breaking when producers add fields, then promote stable paths to typed columns in silver.
- **Python data sources:** an internal REST API or vendor SDK with no maintained connector.

**Watch out for**
- Spark Connect has no RDDs, no `sparkContext` and no JVM internals such as `df._jdf`.
- Check that your table format and every engine reading it support `VARIANT` before storing it.
- A maintained JDBC, Kafka or file connector is faster and better tested than a hand-written Python source.

### Example: query JSON with `VARIANT`

`parse_json` turns JSON text into a `VARIANT` once. `variant_get` extracts a path and casts it;
`try_variant_get` returns null instead of failing when the value does not fit.

In [ ]:
raw_events = spark.createDataFrame(
    [
        (1, '{"device": "ios", "cart": {"items": 3, "coupon": "WELCOME10"}}'),
        (2, '{"device": "web", "cart": {"items": 1}}'),
        (3, '{"device": "android", "cart": {"items": 5, "coupon": null}, "beta": true}'),
    ],
    "event_id int, payload string",
)
events_variant = raw_events.select("event_id", F.parse_json("payload").alias("payload"))

flattened = events_variant.select(
    "event_id",
    F.variant_get("payload", "$.device", "string").alias("device"),
    F.variant_get("payload", "$.cart.items", "int").alias("items"),
    F.try_variant_get("payload", "$.cart.coupon", "string").alias("coupon"),
).orderBy("event_id")
flattened.show()

**What to notice:** `payload` is stored as `variant`. Event 2 has no coupon and event 3 has a null one; both come back as
null. Event 3 also carries a `beta` field that nobody declared, and nothing broke.

### Example: a Python data source

A Python data source subclasses `DataSource` (name and schema) and `DataSourceReader` (one `read` call per
partition). Each `InputPartition` becomes one task. The classes are pickled and run on executors, so import
heavy libraries inside `read` and keep credentials out of the class body.

In [ ]:
from pyspark.sql.datasource import DataSource, DataSourceReader, InputPartition


class FxRatesDataSource(DataSource):
    """Daily FX rates; a real source would call an API inside read()."""

    @classmethod
    def name(cls):
        return "fx_rates"

    def schema(self):
        return "currency string, rate_to_usd double"

    def reader(self, schema):
        return FxRatesReader()


class FxRatesReader(DataSourceReader):
    def partitions(self):
        return [InputPartition(currency) for currency in ("EUR", "GBP", "INR")]

    def read(self, partition):
        rates = {"EUR": 1.08, "GBP": 1.27, "INR": 0.012}  # stand-in for one API call per partition
        yield (partition.value, rates[partition.value])

spark.dataSource.register(FxRatesDataSource)
fx_rates = spark.read.format("fx_rates").load()
fx_rates.orderBy("currency").show()
print("partitions (one per InputPartition):", fx_rates.rdd.getNumPartitions())

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** What runs on the driver and what runs on the executors?

**A.** The driver holds the `SparkSession`, builds and optimizes the plan, schedules tasks and receives anything you `collect()`. Executors hold partitions, run the tasks, and keep shuffle files, cached blocks and broadcast tables.

**Q2.** Why does `groupBy` create a new stage while `filter` does not?

**A.** `filter` decides each row on its own inside its partition, so it is narrow. `groupBy` needs every row with the same key in one place, so rows are shuffled between executors, and every shuffle is a stage boundary.

**Q3.** You call `count()` and then `write` on the same DataFrame. How many times is the source read?

**A.** Twice. Each action runs the whole lineage from the source unless the DataFrame was cached and materialized first.

**Q4.** Where in a plan do you check that a filter reached the file reader?

**A.** In the scan node: `PushedFilters` for predicates the reader evaluates, `PartitionFilters` for directories pruned at plan time, and `ReadSchema` for the columns read.

**Q5.** When would you choose DuckDB or pandas instead of Spark?

**A.** When the data fits on one machine and there is no streaming or heavy parallel Python. A single-node engine has no cluster to operate and is faster to write and debug.

**Q6.** What do you lose with Spark Connect?

**A.** RDDs, `sparkContext`, static configuration and JVM internals such as `df._jdf`. DataFrame and SQL code work unchanged.

## Key takeaways

- A DataFrame is a plan. Nothing runs until an action such as `count`, `show` or a write.
- Narrow operations stay in one stage; wide operations shuffle, and every shuffle is a stage boundary.
- Prefer DataFrame functions to RDDs. Catalyst can optimize them; Python lambdas hide the logic from it.
- Read `explain` output before you tune anything.
- Spark Connect runs the same DataFrame code against a remote server, without RDDs or `sparkContext`.

**Next:** [PySpark 2: reading, SQL and cleaning](pyspark-02-reading-sql-and-cleaning.ipynb)